# Churn Prediction Modeling

This notebook trains and compares churn prediction models for the Career Growth Analytics MVP.

Key constraints:
- Features are computed only from events before the prediction cutoff (signup + 7 days).
- Labels are derived from user actions during days 8-21 after signup.
- Data is split chronologically: 60% train, 20% validation, 20% test.
- Model selection uses validation PR-AUC.
- The operating threshold is chosen on the validation set (F1 by default).

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import sys

sys.path.insert(0, '../src')

from career_growth.features.model_features import prepare_model_matrix
from career_growth.modeling.evaluate import compute_calibration_data, compute_metrics
from career_growth.modeling.explain import (
    compute_permutation_importance,
    extract_logistic_coefficients,
)
from career_growth.modeling.split import chronological_split
from career_growth.modeling.train import train_and_select_model

%matplotlib inline

In [ ]:
data_dir = Path('../data')
users = pd.read_csv(data_dir / 'sample' / 'users.csv')
events = pd.read_csv(data_dir / 'sample' / 'events.csv')
experiment_assignments = pd.read_csv(data_dir / 'sample' / 'experiment_assignments.csv')
labels = pd.read_csv(data_dir / 'processed' / 'labels.csv')

print(f"Users: {len(users):,}")
print(f"Events: {len(events):,}")
print(f"Churn rate: {labels['is_churned'].mean():.2%}")

## Feature Engineering

In [ ]:
model_matrix = prepare_model_matrix(
    users, events, labels, experiment_assignments
)
print(f"Model matrix shape: {model_matrix.shape}")
print(f"Features: {list(model_matrix.columns)}")

## Chronological Split

In [ ]:
train_df, val_df, test_df = chronological_split(
    model_matrix, train_frac=0.6, val_frac=0.2
)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Train churn rate: {train_df['is_churned'].mean():.2%}")
print(f"Val churn rate: {val_df['is_churned'].mean():.2%}")
print(f"Test churn rate: {test_df['is_churned'].mean():.2%}")

## Train and Select Model

In [ ]:
result = train_and_select_model(
    train_df, val_df, test_df, threshold_criterion="f1", random_state=42
)
print(f"Selected model: {result.model_name}")
print(f"Operating threshold: {result.threshold:.4f}")
print("\nValidation metrics:")
for k, v in result.val_metrics.items():
    print(f"  {k}: {v:.4f}")
print("\nTest metrics:")
for k, v in result.test_metrics.items():
    print(f"  {k}: {v:.4f}")

## Precision-Recall and ROC Curves

In [ ]:
from sklearn import metrics

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

precision, recall, _ = metrics.precision_recall_curve(
    test_df['is_churned'], result.test_probabilities
)
axes[0].plot(recall, precision, label=f"PR-AUC = {result.test_metrics['pr_auc']:.4f}")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

fpr, tpr, _ = metrics.roc_curve(test_df['is_churned'], result.test_probabilities)
axes[1].plot(fpr, tpr, label=f"ROC-AUC = {result.test_metrics['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## Calibration

In [ ]:
calibration = compute_calibration_data(
    test_df['is_churned'].to_numpy(), result.test_probabilities
)

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly calibrated")
plt.plot(
    calibration["mean_predicted"],
    calibration["mean_observed"],
    marker="o",
    label="Model",
)
plt.xlabel("Mean Predicted Probability")
plt.ylabel("Fraction of Positives")
plt.title("Reliability Diagram")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

## Explainability

In [ ]:
feature_cols = result.feature_columns
X_val = val_df[feature_cols]
perm_importance = compute_permutation_importance(
    result.model, X_val, val_df['is_churned'].to_numpy(), n_repeats=10, random_state=42
)

top_features = perm_importance.head(15).sort_values("importance_mean")
plt.figure(figsize=(7, 7))
plt.barh(
    top_features["feature"],
    top_features["importance_mean"],
    xerr=top_features["importance_std"],
)
plt.xlabel("Permutation Importance (PR-AUC)")
plt.title("Top Feature Importances (Validation Set)")
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
if result.model_name == "logistic_regression":
    coef_df = extract_logistic_coefficients(result.model)
    display(coef_df.head(15))

## Save Artifacts

In [ ]:
artifact_dir = Path('../artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(result.model, artifact_dir / "churn_model.joblib")

metadata = {
    "model_name": result.model_name,
    "threshold": result.threshold,
    "feature_columns": result.feature_columns,
    "val_metrics": result.val_metrics,
    "test_metrics": result.test_metrics,
}
with open(artifact_dir / "notebook_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Artifacts saved to ../artifacts/")